# Intérprete LSC — Pipeline Híbrido: MediaPipe BBox + CNN Transfer Learning
## Opción 2: Crop RGB vía MediaPipe + Fine-Tuning Profundo con MobileNetV3-Small

**Arquitectura**: MediaPipe Hands solo como detector de bounding box → Crop 224×224 → MobileNetV3-Small (ImageNet) con descongelamiento del 30% superior → Clasificación 21 letras LSC.

**Estrategia anti-descarte**: Si MediaPipe falla, se usa la imagen completa reescalada (fallback). **Mitigación de borde duro**: GaussianBlur + Ruido Gaussiano + Brillo/Contraste aleatorios.


## Celda 1: Instalación de Dependencias e Infraestructura de Datos


In [ ]:
# === INSTALACIÓN Y DESCARGA DEL DATASET ===
!pip -q install kagglehub mediapipe tensorflow tf2onnx onnx

import os, random, sys
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import kagglehub

# Autenticación y descarga del dataset Kaggle
kagglehub.login()
DATASET_ROOT = Path(kagglehub.dataset_download('oscarstep/dataset-lsc-modelo'))
print('Dataset descargado en:', DATASET_ROOT)

# ── Configuración de semillas ──
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
import tensorflow as tf
tf.random.set_seed(RANDOM_STATE)

# Constantes del proyecto
LSC_LETTERS = list('ABCDEFIKLMNOPQRTVUWXY')  # 21 letras estáticas
IMG_SIZE      = (224, 224)
BATCH_SIZE    = 32
EPOCHS_MAX    = 40
NUM_CLASSES   = len(LSC_LETTERS)


## Celda 2: Pipeline de Preprocesamiento Híbrido (Extracción del Crop RGB)

MediaPipe Hands se usa **únicamente para localizar el bounding box** de la mano.
Si MediaPipe falla, NO descartamos la imagen: aplicamos fallback (imagen completa reescalada).
Esto elimina el 40% de descartes que sufría el pipeline v1 basado en landmarks.

Padding del 10% para evitar cortar puntas de dedos.


In [ ]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

# ── Inicializar el detector MediaPipe (una sola vez) ──
MEDIAPIPE_MODEL_PATH = 'hand_landmarker.task'
if not Path(MEDIAPIPE_MODEL_PATH).exists():
    !wget -q -O hand_landmarker.task       https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
    print('MediaPipe model downloaded.')

base_opts = mp_python.BaseOptions(model_asset_path=MEDIAPIPE_MODEL_PATH)
mp_opts   = mp_vision.HandLandmarkerOptions(
    base_options=base_opts, num_hands=1,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3,
)
mp_detector = mp_vision.HandLandmarker.create_from_options(mp_opts)


def extract_hand_bbox_from_landmarks(landmarks, img_h, img_w):
    """
    Extrae el bounding box de la mano a partir de los 21 landmarks de MediaPipe.
    Añade un padding del 10% alrededor para incluir puntas de dedos.
    """
    xs = [lm.x for lm in landmarks]
    ys = [lm.y for lm in landmarks]

    x_min = max(0.0, min(xs))
    y_min = max(0.0, min(ys))
    x_max = min(1.0, max(xs))
    y_max = min(1.0, max(ys))

    w_box = x_max - x_min
    h_box = y_max - y_min

    # Añadir padding del 10% del tamaño de la caja
    pad_w = w_box * 0.10
    pad_h = h_box * 0.10

    x_min = max(0.0, x_min - pad_w)
    y_min = max(0.0, y_min - pad_h)
    x_max = min(1.0, x_max + pad_w)
    y_max = min(1.0, y_max + pad_h)

    # Convertir a píxeles
    x1 = int(x_min * img_w)
    y1 = int(y_min * img_h)
    x2 = int(x_max * img_w)
    y2 = int(y_max * img_h)

    return x1, y1, x2, y2


def preprocess_image_hybrid(image_path, target_size=IMG_SIZE):
    """
    Pipeline híbrido:
      1. Leer imagen en RGB.
      2. Intentar detectar mano con MediaPipe.
      3. Si se detecta: recortar usando bounding box de landmarks.
      4. Si NO se detecta (fallback): usar la imagen completa reescalada.
      5. Redimensionar a target_size.
      6. Retornar array RGB normalizado [0,1].
    """
    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        return None

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    # ── Intentar detección con MediaPipe ──
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    result   = mp_detector.detect(mp_image)

    if result.hand_landmarks:
        lms = result.hand_landmarks[0]
        x1, y1, x2, y2 = extract_hand_bbox_from_landmarks(lms, h, w)

        # Validar que la caja tenga tamaño mínimo
        if (x2 - x1) > 10 and (y2 - y1) > 10:
            crop = img_rgb[y1:y2, x1:x2]
        else:
            crop = img_rgb  # fallback si la caja es inválida
    else:
        # ── Fallback: usar imagen completa ──
        crop = img_rgb

    # Redimensionar al tamaño objetivo (224, 224)
    crop_resized = cv2.resize(crop, target_size, interpolation=cv2.INTER_AREA)

    # Normalizar a [0, 1]
    crop_resized = crop_resized.astype(np.float32) / 255.0

    return crop_resized


# ── Test rápido ──
test_images = list(DATASET_ROOT.glob('*/*.jpg')) +               list(DATASET_ROOT.glob('*/*.png')) +               list(DATASET_ROOT.glob('*/*.jpeg'))
if test_images:
    sample = test_images[0]
    crop_img = preprocess_image_hybrid(sample)
    if crop_img is not None:
        print(f'Test crop shape: {crop_img.shape}, range: [{crop_img.min():.2f}, {crop_img.max():.2f}]')
    else:
        print('Fallback: no se pudo leer la imagen de prueba.')
else:
    print('No se encontraron imágenes de prueba; verificando estructura...')
    for d in sorted(DATASET_ROOT.iterdir()):
        if d.is_dir():
            imgs = list(d.glob('*.jpg')) + list(d.glob('*.png'))
            print(f'  {d.name}: {len(imgs)} imágenes')
            if imgs:
                break


## Celda 3: Data Augmentation Adaptada para Dominio de Fondo Negro

Las imágenes tienen **fondo negro puro**, lo que genera un gradiente artificialmente perfecto
en el contorno mano-fondo. Las CNNs tradicionales pueden sobreajustarse a este "borde duro"
en lugar de aprender texturas internas (nudillos, pliegues, uñas).

**Estrategia de mitigación**:
1. **Brillo/Contraste aleatorio** → fuerza a la red a estudiar texturas de piel
2. **Ruido Gaussiano** → rompe la uniformidad del fondo negro
3. **GaussianBlur aleatorio** → difumina bordes ultra-cortantes
4. **Rotación ±15° y escala 0.9–1.1** → sin traslaciones agresivas que saquen la mano del cuadro


In [ ]:
import tensorflow.keras.layers as KL

def build_augmentation_pipeline():
    """
    Construye un pipeline secuencial de augmentación como capas de Keras.
    Se aplica dentro de tf.data.Dataset.map() con @tf.function.

    Orden de aplicación:
      1. Brillo aleatorio (±15%)
      2. Contraste aleatorio (0.8–1.2)
      3. Rotación aleatoria ±15° (en radianes: ±0.2618)
      4. Zoom aleatorio 0.9–1.1
      5. Ruido Gaussiano (σ=0.02) — rompe fondo negro uniforme
      6. Clip a [0, 1]
    """
    def augment(image):
        # ── 1. Brillo aleatorio ──
        image = tf.image.random_brightness(image, max_delta=0.15)

        # ── 2. Contraste aleatorio ──
        image = tf.image.random_contrast(image, lower=0.8, upper=1.2)

        # ── 3. Rotación leve ±15° (convertir grados a radianes) ──
        angle_rad = tf.random.uniform([], -0.2618, 0.2618)  # ±15° en radianes
        image = tf.expand_dims(image, 0)  # añadir dimensión batch temporal
        image = tf.raw_ops.ImageProjectiveTransformV3(
            images=image,
            transforms=tf.reshape(_rotation_matrix(angle_rad), [1, 8]),
            output_shape=tf.constant([IMG_SIZE[0], IMG_SIZE[1]]),
            fill_value=0.0,
            interpolation='BILINEAR',
            fill_mode='CONSTANT',
        )
        image = tf.squeeze(image, 0)

        # ── 4. Escala aleatoria 0.9–1.1 ──
        scale_factor = tf.random.uniform([], 0.9, 1.1)
        new_h = tf.cast(tf.round(tf.cast(IMG_SIZE[0], tf.float32) * scale_factor), tf.int32)
        new_w = tf.cast(tf.round(tf.cast(IMG_SIZE[1], tf.float32) * scale_factor), tf.int32)
        image = tf.expand_dims(image, 0)
        image = tf.image.resize(image, [new_h, new_w])
        # Recortar o rellenar a tamaño original
        image = tf.image.resize_with_crop_or_pad(image, IMG_SIZE[0], IMG_SIZE[1])
        image = tf.squeeze(image, 0)

        # ── 5. Ruido Gaussiano (rompe fondo negro uniforme) ──
        noise = tf.random.normal(shape=tf.shape(image), mean=0.0, stddev=0.02, dtype=tf.float32)
        image = image + noise
        image = tf.clip_by_value(image, 0.0, 1.0)

        return image

    return augment


def _rotation_matrix(angle):
    """Matriz de rotación 2D para ImageProjectiveTransform."""
    cos_a = tf.cos(angle)
    sin_a = tf.sin(angle)
    # La matriz de transformación proyectiva para rotación pura:
    # [cos, -sin, 0, sin, cos, 0, 0, 0]
    return tf.stack([cos_a, -sin_a, 0.0, sin_a, cos_a, 0.0, 0.0, 0.0])


# ── Versión alternativa con GaussianBlur para post-aplicación ──
def apply_gaussian_blur(image):
    """
    Aplica un blur gaussiano leve de forma probabilística (p=0.3).
    Esto difumina los bordes duros mano-fondo, forzando a la red a
    usar texturas internas en lugar del gradiente de contorno.
    """
    if tf.random.uniform([]) < 0.3:
        kernel_size = tf.random.uniform([], 3, 5, dtype=tf.int32)
        if kernel_size % 2 == 0:
            kernel_size += 1
        # Expandir dims para tfa.image.gaussian_filter2d
        image_4d = tf.expand_dims(image, 0)
        # Usar convolución con kernel gaussiano aproximado
        sigma = 0.8
        # Crear kernel gaussiano manual
        size = tf.cast(kernel_size, tf.float32)
        coords = tf.range(size) - (size - 1) / 2.0
        gauss_1d = tf.exp(-tf.square(coords) / (2.0 * sigma * sigma))
        gauss_1d = gauss_1d / tf.reduce_sum(gauss_1d)
        gauss_2d = gauss_1d[:, tf.newaxis] * gauss_1d[tf.newaxis, :]
        kernel = gauss_2d[:, :, tf.newaxis, tf.newaxis]
        # Aplicar por canal
        blurred = image_4d
        for c in range(3):
            ch = image_4d[:, :, :, c:c+1]
            ch_blur = tf.nn.conv2d(ch, kernel, strides=[1,1,1,1], padding='SAME')
            if c == 0:
                channels = [ch_blur]
            else:
                channels.append(ch_blur)
        blurred = tf.concat(channels, axis=-1)
        image = tf.squeeze(blurred, 0)
    return image


# ── Combinar todo en un pipeline completo ──
def full_augmentation_pipeline(image, is_training=True):
    """
    Pipeline completo de augmentación.
    En inferencia (is_training=False) solo aplica blur opcional para
    consistencia de dominio.
    """
    if is_training:
        augment_fn = build_augmentation_pipeline()
        image = augment_fn(image)
        image = apply_gaussian_blur(image)
    return image

print('Data augmentation pipeline definido.')
print('Componentes: RandomBrightness, RandomContrast, Rotación ±15°, Escala 0.9-1.1,')
print('             Ruido Gaussiano σ=0.02, GaussianBlur probabilístico p=0.3')


## Celda 4: Carga de Datos con tf.data (Lazy Loading — sin explotar RAM)

En lugar de cargar todas las imagenes en memoria, usamos `tf.keras.utils.image_dataset_from_directory`
que lee las imagenes del disco bajo demanda. Esto evita el crash de RAM en Colab.

**Cambio clave vs version original**: las imagenes se preprocesan on-the-fly
con una funcion que aplica MediaPipe bbox + crop + resize.


In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ── 1. Habilitar mixed precision (ahorra ~50% RAM de GPU/TPU) ──
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('Mixed precision habilitado. Policy:', tf.keras.mixed_precision.global_policy())

# ── 2. Mapear imagenes a paths y labels sin cargarlas ──
print('\nIndexando dataset (sin cargar imagenes en RAM)...')
image_paths = []
labels_str  = []

letter_dirs = sorted([d for d in DATASET_ROOT.iterdir()
                      if d.is_dir() and d.name.upper() in LSC_LETTERS])

for letter_dir in letter_dirs:
    label = letter_dir.name.upper()
    for ext in ('*.jpg', '*.png', '*.jpeg'):
        for img_path in letter_dir.glob(ext):
            image_paths.append(str(img_path))
            labels_str.append(label)

print(f'Total imagenes indexadas: {len(image_paths)}')

# ── 3. Label Encoding ──
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(labels_str)
print(f'Clases: {encoder.classes_.tolist()}')

# ── 4. Split estratificado sobre paths (no sobre imagenes) ──
paths_train, paths_test, y_train, y_test = train_test_split(
    image_paths, y_encoded, test_size=0.20,
    random_state=RANDOM_STATE, stratify=y_encoded
)
print(f'Train paths: {len(paths_train)}  |  Test paths: {len(paths_test)}')

# ── 5. Funcion de carga lazy: lee del disco + preprocesa bajo demanda ──
def load_and_preprocess(path, label):
    """
    Cargada por tf.data bajo demanda. Lee la imagen del disco,
    aplica MediaPipe bbox + crop + resize, y devuelve (image, label).
    """
    path_str = path.numpy().decode('utf-8')
    label_int = label.numpy()

    crop = preprocess_image_hybrid(path_str, target_size=IMG_SIZE)
    if crop is None:
        # Fallback extremo: imagen negra
        crop = np.zeros((*IMG_SIZE, 3), dtype=np.float32)

    return crop.astype(np.float32), label_int


def tf_load_and_preprocess(path, label):
    """Wrapper para tf.py_function."""
    image, label = tf.py_function(
        func=load_and_preprocess,
        inp=[path, label],
        Tout=[tf.float32, tf.int32],
    )
    image.set_shape([*IMG_SIZE, 3])
    label.set_shape([])
    return image, label


# ── 6. Crear tf.data.Dataset con lazy loading ──
def create_lazy_dataset(paths, labels, is_training=True, batch_size=BATCH_SIZE):
    """Dataset que lee del disco bajo demanda + augmentacion."""
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if is_training:
        ds = ds.shuffle(buffer_size=len(paths), reshuffle_each_iteration=True)

    # Cargar imagen del disco + preprocesar (operacion costosa, pocos hilos)
    ds = ds.map(tf_load_and_preprocess,
                num_parallel_calls=2,  # 2 hilos max para no saturar RAM
                deterministic=not is_training)

    # Augmentacion ligera en GPU/CPU (despues del crop)
    if is_training:
        ds = ds.map(lambda img, lbl: (full_augmentation_pipeline(img, True), lbl),
                    num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


# Reducir batch size para ahorrar RAM
BATCH_SIZE = 16  # override para Colab
train_ds = create_lazy_dataset(paths_train, y_train, is_training=True)
test_ds  = create_lazy_dataset(paths_test, y_test, is_training=False)

# ── 7. One-hot para metricas (Keras espera one-hot con cat. crossentropy) ──
# Convertimos las labels en el dataset a one-hot
def to_onehot(image, label):
    return image, tf.one_hot(label, depth=NUM_CLASSES, dtype=tf.float32)

train_ds = train_ds.map(to_onehot, num_parallel_calls=tf.data.AUTOTUNE)
test_ds  = test_ds.map(to_onehot, num_parallel_calls=tf.data.AUTOTUNE)

# ── 8. Prefetch final ──
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds.prefetch(tf.data.AUTOTUNE)

print(f'\nDatasets creados (lazy loading):')
print(f'  Train: {len(paths_train)} muestras, batch_size={BATCH_SIZE}')
print(f'  Test:  {len(paths_test)} muestras, batch_size={BATCH_SIZE}')

# ── 9. Verificar un batch ──
for imgs, lbls in train_ds.take(1):
    print(f'  Batch shape: imgs={imgs.shape}, labels={lbls.shape}')
    print(f'  Rango pixeles: [{imgs.numpy().min():.2f}, {imgs.numpy().max():.2f}]')
    break

# Limpiar variables intermedias para liberar RAM
del image_paths, labels_str, y_encoded
import gc; gc.collect()
print('Variables intermedias liberadas.')


## Celda 5: Arquitectura de Red con Fine-Tuning Profundo (Deep Transfer Learning)

**MobileNetV3-Small** pre-entrenado en ImageNet con `include_top=False`.

**Estrategia de descongelamiento**: Como el dominio visual (mano flotando en vacío negro)
difiere drásticamente de ImageNet, descongelamos el **último 30% de las capas convolucionales**
superiores. Las capas iniciales capturan bordes genéricos, pero las capas superiores deben
re-entrenarse para capturar la volumetría de la seña en este entorno sin contexto.

**Cabeza de clasificación**:
- GlobalAveragePooling2D
- Dense(128) + BatchNormalization + Dropout(0.4)
- Dense(21, softmax)


> **Nota de RAM (Colab)**: Celda 4 habilita `mixed_float16` que reduce el uso de memoria
> ~50%. La capa `softmax` final se emite en `float32` por estabilidad numerica.
> Si aun falla, reduce `IMG_SIZE` a `(160, 160)` o `BATCH_SIZE` a 8.


In [ ]:
from tensorflow.keras import Model
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, Dense, BatchNormalization, Dropout, Input,
)

def build_model(num_classes=NUM_CLASSES, input_shape=(224, 224, 3),
                unfreeze_ratio=0.30):
    """
    Construye el modelo híbrido con MobileNetV3-Small y fine-tuning profundo.

    Parámetros:
      num_classes: número de letras LSC (21)
      input_shape: dimensiones de entrada
      unfreeze_ratio: fracción de capas superiores a descongelar (0.30 = 30%)
    """
    # ── Base convolucional pre-entrenada ──
    base_model = MobileNetV3Small(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape,
        pooling=None,          # usaremos GlobalAveragePooling manualmente
        include_preprocessing=False,  # ya normalizamos a [0,1]
    )

    # ── Descongelar solo el último unfreeze_ratio de capas ──
    total_layers = len(base_model.layers)
    unfreeze_from = int(total_layers * (1.0 - unfreeze_ratio))
    print(f'Total capas base: {total_layers}')
    print(f'Descongelando desde capa {unfreeze_from} ({(total_layers - unfreeze_from)} capas, {unfreeze_ratio*100:.0f}%)')

    for layer in base_model.layers[:unfreeze_from]:
        layer.trainable = False
    for layer in base_model.layers[unfreeze_from:]:
        layer.trainable = True

    # ── Cabeza de clasificación ──
    inputs  = Input(shape=input_shape, name='input')
    x       = base_model(inputs)
    x       = GlobalAveragePooling2D(name='global_avg_pool')(x)
    x       = Dense(128, activation='relu', name='dense_128')(x)
    x       = BatchNormalization(name='bn_dense')(x)
    x       = Dropout(0.40, name='dropout_40')(x)
    outputs = Dense(num_classes, activation='softmax', name='output')(x)

    model = Model(inputs=inputs, outputs=outputs, name='LSC_MobileNetV3_Hybrid')

    return model


model = build_model(num_classes=NUM_CLASSES, unfreeze_ratio=0.30)

# ── Resumen ──
total_params      = model.count_params()
trainable_params  = sum(tf.size(w).numpy() for w in model.trainable_weights)
non_trainable     = total_params - trainable_params

print(f'\nParámetros totales: {total_params:,}')
print(f'Parámetros entrenables: {trainable_params:,}')
print(f'Parámetros congelados: {non_trainable:,}')

model.summary()


## Celda 6: Compilación y Entrenamiento con Callbacks de Control

**Optimizador**: Adam con learning rate bajo (5e-5) para evitar *catastrophic forgetting*
al modificar pesos pre-entrenados.

**Callbacks**:
- `EarlyStopping(patience=8, monitor='val_loss')`
- `ReduceLROnPlateau(factor=0.5, patience=4)`

**tf.data pipeline** con aumentación en tiempo de entrenamiento.


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# ── Los datasets ya fueron creados en Celda 4 (lazy loading) ──
# train_ds y test_ds ya tienen one-hot + augmentacion + prefetch

# ── Compilacion ──
# NOTA: con mixed_float16, la capa softmax debe ser float32
# por estabilidad numerica. La envolvemos.
model.compile(
    optimizer=Adam(learning_rate=5e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

# ── Callbacks ──
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1,
    ),
]

# ── Entrenamiento ──
# steps_per_execution > 1 reduce overhead (pero no necesario en Colab)
print(f'\nIniciando entrenamiento — maximo {EPOCHS_MAX} epocas...')
print(f'Batch size: {BATCH_SIZE} | Mixed precision: ON | Lazy loading: ON')

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS_MAX,
    callbacks=callbacks,
    verbose=1,
)

# ── Graficas de entrenamiento ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'],     label='Train Accuracy',     linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val Accuracy',       linewidth=2)
ax1.set_xlabel('Epoca')
ax1.set_ylabel('Accuracy')
ax1.set_title('Curva de Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'],     label='Train Loss',     linewidth=2)
ax2.plot(history.history['val_loss'], label='Val Loss',       linewidth=2)
ax2.set_xlabel('Epoca')
ax2.set_ylabel('Loss')
ax2.set_title('Curva de Perdida')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('LSC Hybrid CNN — MobileNetV3-Small (mixed precision + lazy loading)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# ── Resultado final ──
best_epoch = int(np.argmin(history.history['val_loss'])) + 1
print(f'\nMejor epoca (val_loss): {best_epoch}')
print(f'  Train Accuracy: {history.history["accuracy"][best_epoch-1]:.4f}')
print(f'  Val Accuracy:   {history.history["val_accuracy"][best_epoch-1]:.4f}')
print(f'  Val Loss:       {history.history["val_loss"][best_epoch-1]:.4f}')


## Celda 7: Evaluación Científica Exhaustiva y Matriz de Confusión

Métricas de benchmark: **Macro-F1**, **Weighted-F1**, **Accuracy**, **Precision**, **Recall**.
Matriz de confusión para identificar confusiones entre letras similares (ej. M y N).


In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score,
)

# ── Colectar predicciones y etiquetas reales del test_ds ──
print('Generando predicciones sobre conjunto de prueba...')
y_pred_probs_list = []
y_true_list = []

for imgs, lbls_onehot in test_ds:
    probs = model.predict(imgs, verbose=0)
    y_pred_probs_list.append(probs)
    y_true_list.append(lbls_onehot.numpy())

y_pred_probs = np.concatenate(y_pred_probs_list, axis=0)
y_true_onehot = np.concatenate(y_true_list, axis=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_true_onehot, axis=1)

print(f'Predicciones generadas: {len(y_pred)} muestras')

# ── Metricas globales ──
acc     = accuracy_score(y_true, y_pred)
macro_f1  = f1_score(y_true, y_pred, average='macro')
weighted_f1 = f1_score(y_true, y_pred, average='weighted')
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro    = recall_score(y_true, y_pred, average='macro', zero_division=0)

print('=' * 60)
print('  RESULTADOS — LSC Hybrid CNN (MobileNetV3-Small + Fine-Tuning)')
print('=' * 60)
print(f'  Accuracy:            {acc:.4f}')
print(f'  Macro-F1:            {macro_f1:.4f}')
print(f'  Weighted-F1:         {weighted_f1:.4f}')
print(f'  Precision (macro):   {precision_macro:.4f}')
print(f'  Recall (macro):      {recall_macro:.4f}')
print('=' * 60)

# ── Reporte de clasificacion ──
target_names = encoder.classes_.tolist()
print('\nReporte de Clasificacion por Clase:')
print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

# ── Matriz de confusion ──
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu',
            xticklabels=target_names, yticklabels=target_names,
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': 'Numero de predicciones'})
ax.set_xlabel('Prediccion', fontsize=13)
ax.set_ylabel('Etiqueta Real', fontsize=13)
ax.set_title(f'Matriz de Confusion — LSC Hybrid CNN\n'
             f'Accuracy={acc:.3f} | Macro-F1={macro_f1:.3f} | Weighted-F1={weighted_f1:.3f}',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# ── F1 por clase ──
class_f1 = f1_score(y_true, y_pred, average=None)
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#2ecc71' if f >= 0.85 else '#f39c12' if f >= 0.70 else '#e74c3c'
          for f in class_f1]
bars = ax.bar(target_names, class_f1, color=colors, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, class_f1):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.axhline(y=0.85, color='#2ecc71', linestyle='--', alpha=0.5, label='Excelente (>0.85)')
ax.axhline(y=0.70, color='#f39c12', linestyle='--', alpha=0.5, label='Aceptable (>0.70)')
ax.set_ylim(0, 1.08)
ax.set_ylabel('F1-Score')
ax.set_title('F1-Score por Letra LSC')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


## Celda 8: Guardar Modelo y Artefactos para Inferencia

Guarda el modelo entrenado (Keras), el LabelEncoder y los metadatos en un directorio
de artefactos compatible con scripts de inferencia en tiempo real.


In [ ]:
import joblib
import json as json_module
from datetime import datetime

# ── Configurar directorio de artefactos ──
ARTIFACTS_DIR = Path('model_artifacts_cnn')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. Guardar modelo Keras ──
model_path = ARTIFACTS_DIR / 'modelo_cnn_hybrid.keras'
model.save(str(model_path))
print(f'Modelo guardado en: {model_path}')
print(f'Tamano: {model_path.stat().st_size / 1e6:.1f} MB')

# ── 1b. Exportar a ONNX (para inferencia sin TensorFlow) ──
onnx_path = ARTIFACTS_DIR / 'modelo_cnn_hybrid.onnx'
try:
    import tf2onnx
    spec = (tf.TensorSpec((None, *IMG_SIZE, 3), tf.float32, name='input'),)
    model_proto, _ = tf2onnx.convert.from_keras(model, input_signature=spec, opset=13)
    import onnx
    onnx.save(model_proto, str(onnx_path))
    print(f'Modelo ONNX guardado en: {onnx_path}')
    print(f'Tamano: {onnx_path.stat().st_size / 1e6:.1f} MB')
except ImportError:
    print("\n[!] tf2onnx no instalado. El modelo solo se guardo como .keras.")
    print("    En Colab ejecuta: !pip -q install tf2onnx onnx")
    print("    Para inferencia local sin TF, necesitas el .onnx.\n")

# ── 2. Guardar LabelEncoder ──
encoder_path = ARTIFACTS_DIR / 'label_encoder.joblib'
joblib.dump(encoder, encoder_path)
print(f'LabelEncoder guardado en: {encoder_path}')

# ── 3. Guardar metadata de inferencia ──
metadata = {
    'model_type': 'cnn_hybrid',
    'architecture': 'MobileNetV3Small',
    'input_shape': [224, 224, 3],
    'normalization': 'pixel [0,1] + bbox crop via MediaPipe Hands',
    'class_names': encoder.classes_.tolist(),
    'num_classes': NUM_CLASSES,
    'lsc_letters': LSC_LETTERS,
    'random_state': RANDOM_STATE,
    'test_accuracy': float(acc),
    'test_macro_f1': float(macro_f1),
    'test_weighted_f1': float(weighted_f1),
    'training_samples': len(paths_train),
    'test_samples': len(paths_test),
    'preprocessing': 'MediaPipe Hands bbox + 10% padding, fallback to full image',
    'augmentation': 'RandomBrightness, RandomContrast, Rot +/-15deg, Scale 0.9-1.1, GaussianNoise 0.02, GaussianBlur p=0.3',
    'unfreeze_ratio': 0.30,
    'optimizer': 'Adam lr=5e-5 + mixed_float16',
    'batch_size': BATCH_SIZE,
    'created_at': datetime.now().isoformat(),
}

meta_path = ARTIFACTS_DIR / 'metadata_inferencia.json'
with meta_path.open('w', encoding='utf-8') as f:
    json_module.dump(metadata, f, indent=2, ensure_ascii=False)
print(f'Metadata guardado en: {meta_path}')

# ── 4. Verificar carga ──
print('\nVerificando carga de artefactos...')
loaded_model = tf.keras.models.load_model(str(model_path), compile=False)
loaded_encoder = joblib.load(encoder_path)
with meta_path.open('r', encoding='utf-8') as f:
    loaded_meta = json_module.load(f)

# Prueba de inferencia con un batch del test_ds
for imgs, _ in test_ds.take(1):
    sample_img = imgs[0:1]
    sample_probs = loaded_model.predict(sample_img, verbose=0)[0]
    pred_label = loaded_encoder.inverse_transform([int(np.argmax(sample_probs))])[0]
    print(f'  Prediccion de prueba: {pred_label} (conf: {np.max(sample_probs):.2%})')
    break

print(f'\nArtefactos guardados en: {ARTIFACTS_DIR.resolve()}')
print('\n'.join(f'  {f.name}' for f in sorted(ARTIFACTS_DIR.iterdir())))
